In [1]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

## Задача преобразовать данные из файла bike_share/daily_rent_detail.csv, для работы с гипотезами

- читаю файл bike_share/daily_rent_detail.csv

In [2]:
daily_rent_detail = pd.read_csv('../../bike_share/daily_rent_detail.csv')
daily_rent_detail

C:\Users\lenov\AppData\Local\Temp\ipykernel_17376\2254643125.py:1: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  daily_rent_detail = pd.read_csv('../../bike_share/daily_rent_detail.csv')


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29,2020-05-31 18:25:22,Anacostia Library,31804,11th & H St NE,31614.0,38.865784,-76.978400,38.899983,-76.991383,casual
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04,2020-05-09 15:06:33,10th & E St NW,31256,21st St & Constitution Ave NW,31261.0,38.895914,-77.026064,38.892459,-77.046567,member
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19,2020-05-24 17:43:51,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268.0,38.934267,-77.057979,38.916787,-77.028139,member
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52,2020-05-27 15:47:13,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119.0,38.934267,-77.057979,38.921074,-77.031887,casual
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03,2020-05-31 14:30:30,Georgia Ave & Morton St NW,31419,17th & K St NW,31213.0,38.932128,-77.023500,38.902760,-77.038630,casual
...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,117C27DBDD138B72,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,NaN,NaN,NaN,NaN,38.890000,-77.000000,38.890000,-77.000000,member
16086668,4774F4D630258482,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,NaN,NaN,NaN,NaN,38.900000,-77.000000,38.870000,-76.950000,member
16086669,D75836E25E77B5EC,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,NaN,NaN,NaN,NaN,38.920000,-76.990000,38.920000,-77.000000,member
16086670,3B888603D18116DC,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,NaN,NaN,NaN,NaN,38.920000,-77.020000,38.920000,-77.030000,member


- вывожу информацию что бы постреть типы данных

In [3]:
daily_rent_detail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16086672 entries, 0 to 16086671
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 1.6+ GB


- меняю типы данных для столбцов started_at и ended_at

In [4]:
daily_rent_detail['started_at'] = pd.to_datetime(daily_rent_detail['started_at'], format='mixed')
daily_rent_detail['ended_at'] = pd.to_datetime(daily_rent_detail['ended_at'], format='mixed')

- добавляю столбец duration_in_minutes, он будет нужен для определения продолжительности поездки
- сразу сделаю в минутах

In [5]:
daily_rent_detail['duration_in_minutes'] = daily_rent_detail['ended_at'] - daily_rent_detail['started_at']
daily_rent_detail['duration_in_minutes'] = daily_rent_detail['duration_in_minutes'].dt.total_seconds().div(60)

- смотрю что получилось

In [6]:
daily_rent_detail

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_in_minutes
0,946D42AD89539210,docked_bike,2020-05-30 17:25:29.000,2020-05-31 18:25:22.000,Anacostia Library,31804,11th & H St NE,31614.0,38.865784,-76.978400,38.899983,-76.991383,casual,1499.883333
1,CC46FAAB662B8613,docked_bike,2020-05-09 14:42:04.000,2020-05-09 15:06:33.000,10th & E St NW,31256,21st St & Constitution Ave NW,31261.0,38.895914,-77.026064,38.892459,-77.046567,member,24.483333
2,72F00B2FB833D6ED,docked_bike,2020-05-24 17:27:19.000,2020-05-24 17:43:51.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,12th & U St NW,31268.0,38.934267,-77.057979,38.916787,-77.028139,member,16.533333
3,4DFBE6AED989DF35,docked_bike,2020-05-27 15:29:52.000,2020-05-27 15:47:13.000,Connecticut Ave & Newark St NW / Cleveland Park,31305,14th & Belmont St NW,31119.0,38.934267,-77.057979,38.921074,-77.031887,casual,17.350000
4,1AAFE6B4331AB9DF,docked_bike,2020-05-31 14:06:03.000,2020-05-31 14:30:30.000,Georgia Ave & Morton St NW,31419,17th & K St NW,31213.0,38.932128,-77.023500,38.902760,-77.038630,casual,24.450000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16086667,117C27DBDD138B72,electric_bike,2024-08-01 08:10:22.293,2024-08-01 08:10:37.641,NaN,NaN,NaN,NaN,38.890000,-77.000000,38.890000,-77.000000,member,0.255800
16086668,4774F4D630258482,electric_bike,2024-08-08 10:05:21.938,2024-08-08 10:21:46.654,NaN,NaN,NaN,NaN,38.900000,-77.000000,38.870000,-76.950000,member,16.411933
16086669,D75836E25E77B5EC,electric_bike,2024-08-03 16:29:32.252,2024-08-03 16:35:43.179,NaN,NaN,NaN,NaN,38.920000,-76.990000,38.920000,-77.000000,member,6.182117
16086670,3B888603D18116DC,electric_bike,2024-08-03 02:49:45.661,2024-08-03 02:59:56.877,NaN,NaN,NaN,NaN,38.920000,-77.020000,38.920000,-77.030000,member,10.186933


- нужно убрать отрицательные значения из dataframe daily_rent_detail

In [7]:
daily_rent_detail = daily_rent_detail[daily_rent_detail['duration_in_minutes'] >= 0]

- нужно привести в один тип данных столбцы start_station_id и end_station_id

In [8]:
daily_rent_detail['start_station_id'] = daily_rent_detail['start_station_id'].replace(np.nan, 0)
daily_rent_detail['end_station_id'] = daily_rent_detail['end_station_id'].replace(np.nan, 0)

C:\Users\lenov\AppData\Local\Temp\ipykernel_17376\2403721374.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_rent_detail['start_station_id'] = daily_rent_detail['start_station_id'].replace(np.nan, 0)
C:\Users\lenov\AppData\Local\Temp\ipykernel_17376\2403721374.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_rent_detail['end_station_id'] = daily_rent_detail['end_station_id'].replace(np.nan, 0)


- для того что бы поменять тип данных нужно убрать информацию о станции 'MTL-ECO5-03' (2 строчки данных)

In [9]:
daily_rent_detail = daily_rent_detail[daily_rent_detail['start_station_id'] != 'MTL-ECO5-03']

- нужно убрать из столбца end_lng строки с нулевыми значениями (там их 15)

In [10]:
daily_rent_detail = daily_rent_detail[daily_rent_detail['end_lng'] != 0]

In [12]:
daily_rent_detail = daily_rent_detail.astype(
    {
        'start_station_id': 'float32',
        'end_station_id': 'float32'
    }
)

In [13]:
daily_rent_detail = daily_rent_detail.astype(
    {
        'start_station_id': 'int32',
        'end_station_id': 'int32'
    }
)

In [14]:
daily_rent_detail.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16078038 entries, 0 to 16086671
Data columns (total 14 columns):
 #   Column               Dtype         
---  ------               -----         
 0   ride_id              object        
 1   rideable_type        object        
 2   started_at           datetime64[ns]
 3   ended_at             datetime64[ns]
 4   start_station_name   object        
 5   start_station_id     int32         
 6   end_station_name     object        
 7   end_station_id       int32         
 8   start_lat            float64       
 9   start_lng            float64       
 10  end_lat              float64       
 11  end_lng              float64       
 12  member_casual        object        
 13  duration_in_minutes  float64       
dtypes: datetime64[ns](2), float64(5), int32(2), object(5)
memory usage: 1.7+ GB


- подготовлю dataframe daily_rent_detail для записи в файл parquet

- меняю типы данных для каждого столбца в daily_rent_detail

In [15]:
daily_rent_detail = daily_rent_detail.astype(
    {
        'ride_id': 'object',
        'rideable_type': 'object',
        'started_at': 'datetime64[us]',
        'ended_at': 'datetime64[us]',
        'start_station_name': 'object',
        'start_station_id': 'int32',
        'end_station_name': 'object',
        'end_station_id': 'int32',
        'start_lat': 'float32',
        'start_lng': 'float32',
        'end_lat': 'float32',
        'end_lng': 'float32',
        'member_casual': 'object',
        'duration_in_minutes': 'float32'
    }
)

- записываю dataframe daily_rent_detail в файл parquet

In [16]:
table_for_parquet = pa.Table.from_pandas(daily_rent_detail)

In [17]:
pq.write_table(table_for_parquet, '../../data_after_preprocessing/daily_rent_detail.parquet')